# 06b - Eval: SEA-LION generation model

Follow-up on `06_eval.ipynb`. Identical Ragas harness (Faithfulness,
AnswerRelevancy) and identical retrieval (BGE-M3 + SEA-LION reranker), but
scores answers written by `aisingapore/Llama-SEA-LION-v3-8B-IT` (see
`05b_generation_sealion.ipynb`) instead of `qwen3:8b`.

Same 8-query labeled set as `06_eval.ipynb` (Burmese and Thai dropped there for
judge/generation-noise reasons -- see that notebook's Step 2 comment), so the
faithfulness/answer_relevancy averages here are directly comparable to
`06_eval.ipynb`'s qwen3:8b numbers, query-for-query.

## Step 1: Setup

Same as `06_eval.ipynb` -- chunks, BGE-M3 embeddings, Chroma, BM25, SEA-LION
reranker -- except `OLLAMA_MODEL_NAME` (set in Step 3) points at the SEA-LION
generation model instead of qwen3:8b.

In [1]:
import json
import re
from pathlib import Path

import chromadb
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

CHUNKS_PATH = Path("../data/processed/chunks.json")
EMBEDDINGS_PATH = Path("../data/processed/embeddings_bge_m3.npy")
CHUNK_IDS_PATH = Path("../data/processed/chunk_ids_bge_m3.json")
CHROMA_DIR = Path("../data/processed/chroma")

EMBEDDING_MODEL_NAME = "BAAI/bge-m3"
# Reranker unchanged from 06_eval.ipynb -- retrieval is not a variable in this
# experiment, only the generation model (set in Step 3) is.
RERANKER_MODEL_NAME = "aisingapore/SEA-LION-E5-Embedding-600M"

chunks = json.loads(CHUNKS_PATH.read_text(encoding="utf-8"))
chunks_by_id = {chunk["chunk_id"]: chunk for chunk in chunks}

embeddings = np.load(EMBEDDINGS_PATH)
chunk_ids = json.loads(CHUNK_IDS_PATH.read_text(encoding="utf-8"))
chunk_id_to_idx = {cid: i for i, cid in enumerate(chunk_ids)}

client = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection = client.get_collection(name="bge_m3")

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
reranker_model = SentenceTransformer(RERANKER_MODEL_NAME)


def tokenize(text: str) -> list[str]:
    return re.findall(r"\w+", text.lower())


bm25 = BM25Okapi([tokenize(chunks_by_id[cid]["text"]) for cid in chunk_ids])


def dense_retrieve(query: str, top_k: int = 5) -> list[str]:
    query_embedding = embedding_model.encode(query, convert_to_numpy=True)
    results = collection.query(query_embeddings=[query_embedding.tolist()], n_results=top_k)
    return list(results["ids"][0])


def bm25_retrieve(query: str, top_k: int = 5) -> list[str]:
    scores = bm25.get_scores(tokenize(query))
    ranked = sorted(zip(chunk_ids, scores), key=lambda x: x[1], reverse=True)[:top_k]
    return [cid for cid, _ in ranked]


def hybrid_retrieve(query: str, top_k: int = 5) -> list[str]:
    dense_ids = dense_retrieve(query, top_k=10)
    bm25_ids = bm25_retrieve(query, top_k=10)
    scores: dict[str, float] = {}
    for ranked in (dense_ids, bm25_ids):
        for rank, cid in enumerate(ranked, start=1):
            scores[cid] = scores.get(cid, 0.0) + 1.0 / (60 + rank)
    return [cid for cid, _ in sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k]]


def hybrid_rerank_retrieve(query: str, top_k: int = 5) -> list[str]:
    candidates = hybrid_retrieve(query, top_k=10)
    # STS is the only named prompt documented on the SEA-LION-E5 model card,
    # used for both query and passage since this only needs a symmetric
    # cosine-similarity score for reranking, not asymmetric retrieval.
    query_embedding = reranker_model.encode(query, convert_to_numpy=True, prompt_name="STS")
    candidate_embeddings = reranker_model.encode(
        [chunks_by_id[cid]["text"] for cid in candidates],
        convert_to_numpy=True,
        prompt_name="STS",
    )
    similarities = candidate_embeddings @ query_embedding / (
        np.linalg.norm(candidate_embeddings, axis=1) * np.linalg.norm(query_embedding)
    )
    reranked = sorted(zip(candidates, similarities), key=lambda x: x[1], reverse=True)
    return [cid for cid, _ in reranked[:top_k]]


RETRIEVAL_METHODS = {
    "dense": dense_retrieve,
    "bm25": bm25_retrieve,
    "hybrid": hybrid_retrieve,
    "hybrid_rerank": hybrid_rerank_retrieve,
}

len(chunks), collection.count(), list(RETRIEVAL_METHODS)

c:\Users\rames\Documents\GitHub\migrantBuddy\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 9247.41it/s]


(16, 16, ['dense', 'bm25', 'hybrid', 'hybrid_rerank'])

## Step 2: Labeled eval set

Identical 8-query set to `06_eval.ipynb` (Burmese and Thai dropped there for
judge/generation-noise reasons, not a retrieval-quality issue -- see that
notebook's Step 2 comment for the full explanation).

In [2]:
def find_chunk_id(document_slug: str, heading_contains: str) -> str:
    matches = [
        cid
        for cid in chunk_ids
        if document_slug in chunks_by_id[cid]["url"]
        and heading_contains.lower() in chunks_by_id[cid]["heading_path"].lower()
    ]
    assert len(matches) == 1, f"Expected exactly 1 match for {heading_contains!r}, got {matches}"
    return matches[0]


def find_chunk_id_by_document(document_slug: str) -> str:
    matches = [cid for cid in chunk_ids if document_slug in chunks_by_id[cid]["url"]]
    assert len(matches) == 1, f"Expected exactly 1 chunk for {document_slug!r}, got {matches}"
    return matches[0]


LABELED_QUERIES = [
    {
        "language": "en",
        "text": "How much overtime pay am I entitled to?",
        "correct_chunk_id": find_chunk_id("hours-of-work", "Overtime pay"),
    },
    {
        "language": "en",
        "text": "When must my employer pay my salary?",
        "correct_chunk_id": find_chunk_id("paying-salary", "How often salary must be paid"),
    },
    {
        "language": "ms",
        "text": "Bilakah majikan saya perlu bayar gaji saya?",
        "correct_chunk_id": find_chunk_id("paying-salary", "How often salary must be paid"),
    },
    {
        "language": "ta",
        "text": "எனக்கு எவ்வளவு கூடுதல் நேர ஊதியம் கிடைக்கும்?",
        "correct_chunk_id": find_chunk_id("hours-of-work", "Overtime pay"),
    },
    {
        "language": "vi",
        "text": "Chủ sử dụng lao động của tôi phải trả lương khi nào?",
        "correct_chunk_id": find_chunk_id("paying-salary", "How often salary must be paid"),
    },
    {
        "language": "en",
        "text": "Who pays repatriation costs when my Work Permit ends?",
        "correct_chunk_id": find_chunk_id("work-permit-conditions", "employment ends"),
    },
    {
        "language": "en",
        "text": "How much medical insurance must my employer provide?",
        "correct_chunk_id": find_chunk_id("medical-insurance", "should cover"),
    },
    {
        "language": "en",
        "text": "How can I contact MOM?",
        "correct_chunk_id": find_chunk_id_by_document("contact-us"),
    },
]

for q in LABELED_QUERIES:
    print(f"[{q['language']}] {q['text']}\n  -> {q['correct_chunk_id']}\n")

[en] How much overtime pay am I entitled to?
  -> https-www-mom-gov-sg-employment-practices-hours-of-work-overtime-and-rest-days::chunk-2

[en] When must my employer pay my salary?
  -> https-www-mom-gov-sg-employment-practices-salary-paying-salary::chunk-1

[ms] Bilakah majikan saya perlu bayar gaji saya?
  -> https-www-mom-gov-sg-employment-practices-salary-paying-salary::chunk-1

[ta] எனக்கு எவ்வளவு கூடுதல் நேர ஊதியம் கிடைக்கும்?
  -> https-www-mom-gov-sg-employment-practices-hours-of-work-overtime-and-rest-days::chunk-2

[vi] Chủ sử dụng lao động của tôi phải trả lương khi nào?
  -> https-www-mom-gov-sg-employment-practices-salary-paying-salary::chunk-1

[en] Who pays repatriation costs when my Work Permit ends?
  -> https-www-mom-gov-sg-passes-and-permits-work-permit-for-foreign-worker-sector-specific-rules-work-permit-conditions::chunk-3

[en] How much medical insurance must my employer provide?
  -> https-www-mom-gov-sg-passes-and-permits-work-permit-for-foreign-worker-sector-sp

## Step 3: Ragas setup

`OLLAMA_MODEL_NAME` here is the SEA-LION generation model, not qwen3:8b. Judge
LLM (`llama3.1:8b`) and embeddings (BGE-M3) unchanged from `06_eval.ipynb` --
same judge scoring both generation models keeps the comparison fair.

In [3]:
OLLAMA_MODEL_NAME = "aisingapore/Llama-SEA-LION-v3-8B-IT"
JUDGE_MODEL_NAME = "llama3.1:8b"  # same judge as 06_eval.ipynb -- fair comparison
OLLAMA_BASE_URL = "http://localhost:11434"

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms import LangchainLLMWrapper

judge_llm = LangchainLLMWrapper(ChatOllama(model=JUDGE_MODEL_NAME, base_url=OLLAMA_BASE_URL))
ragas_embeddings = LangchainEmbeddingsWrapper(HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME))

judge_llm, ragas_embeddings

C:\Users\rames\AppData\Local\Temp\ipykernel_18940\3038500055.py:10: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  judge_llm = LangchainLLMWrapper(ChatOllama(model=JUDGE_MODEL_NAME, base_url=OLLAMA_BASE_URL))
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 45215.68it/s]
C:\Users\rames\AppData\Local\Temp\ipykernel_18940\3038500055.py:11: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME))


(LangchainLLMWrapper(langchain_llm=ChatOllama(...)),
 LangchainEmbeddingsWrapper(embeddings=HuggingFaceEmbeddings(...)))

## Step 4: Build the eval dataset

For each labeled query, retrieve -> generate with the SEA-LION generation
model to get real `{question, contexts, answer}` triples for Ragas.

In [4]:
import requests

SYSTEM_PROMPT = """You are migrantBuddy, an assistant that answers questions about \
Singapore employment rules (work passes, salary, working hours) for migrant workers.

Answer ONLY using the information in the provided context. If the context does not \
contain enough information to answer the question, say so clearly instead of \
guessing. Do not use any outside knowledge. Keep answers clear and concise, \
suitable for someone who may not be a native English speaker."""


def build_prompt(query: str, context_chunks: list[dict]) -> str:
    context_text = "\n\n---\n\n".join(
        f"Source: {chunk['url']}\n{chunk['text']}" for chunk in context_chunks
    )
    return f"""Context:
{context_text}

Question: {query}

Answer:"""


def generate_answer(query: str, method: str = "hybrid_rerank", top_k: int = 5) -> dict:
    retrieved_ids = RETRIEVAL_METHODS[method](query, top_k)
    context_chunks = [chunks_by_id[cid] for cid in retrieved_ids]
    user_prompt = build_prompt(query, context_chunks)

    response = requests.post(
        f"{OLLAMA_BASE_URL}/api/chat",
        json={
            "model": OLLAMA_MODEL_NAME,
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
            ],
            "stream": False,
        },
        timeout=120,
    )
    response.raise_for_status()
    answer = response.json()["message"]["content"]

    return {
        "question": query,
        "contexts": [chunk["text"] for chunk in context_chunks],
        "answer": answer,
    }


eval_dataset_rows = [generate_answer(q["text"]) for q in LABELED_QUERIES]
eval_dataset_rows[0]

{'question': 'How much overtime pay am I entitled to?',
 'contexts': ['Hours of work, overtime and rest day > Overtime pay\n\nOvertime work is all work in excess of the normal hours of work (excluding breaks).\n\nYou can claim overtime if you are:\n\n- A non-workman earning a monthly basic salary of $2,600 or less.\n- A workman earning a monthly basic salary of $4,500 or less. \n\nThe overtime rate payable for non-workmen is **capped at the salary level of $2,600, or an hourly rate of $13.60**.\n\nFor overtime work, your employer must pay you **at least 1.5 times** the hourly basic rate of pay. Payment must be made **within 14 days** after the last day of the salary period.\n\nA non-workman earns $2,600 a month and works 2 hours of overtime. The overtime pay is:\n\n$13.60 × 1.5 × 2 hours = $40.80\n\nCalculate your overtime pay\n\nOvertime pay is calculated as follows:\n\n- Hourly basic rate of pay × 1.5 × number of hours worked overtime\n\nThe hourly basic rate of pay is calculated as 

## Step 5: Run Ragas metrics

`Faithfulness` and `AnswerRelevancy`, same as `06_eval.ipynb`.

In [5]:
import asyncio

from ragas import SingleTurnSample
from ragas.metrics import AnswerRelevancy, Faithfulness

samples = [
    SingleTurnSample(
        user_input=row["question"],
        retrieved_contexts=row["contexts"],
        response=row["answer"],
    )
    for row in eval_dataset_rows
]

faithfulness_metric = Faithfulness(llm=judge_llm)
answer_relevancy_metric = AnswerRelevancy(llm=judge_llm, embeddings=ragas_embeddings)

ragas_scores = []
for i, sample in enumerate(samples):
    query_info = LABELED_QUERIES[i]
    row = {
        "language": query_info["language"],
        "query": query_info["text"],
        "answer": eval_dataset_rows[i]["answer"],
    }

    try:
        row["faithfulness"] = await asyncio.wait_for(
            faithfulness_metric.single_turn_ascore(sample), timeout=300
        )
    except Exception as e:
        row["faithfulness"] = None
        print(f"FAILED faithfulness [{query_info['language']}] {type(e).__name__}: {e}")

    try:
        row["answer_relevancy"] = await asyncio.wait_for(
            answer_relevancy_metric.single_turn_ascore(sample), timeout=300
        )
    except Exception as e:
        row["answer_relevancy"] = None
        print(f"FAILED answer_relevancy [{query_info['language']}] {type(e).__name__}: {e}")

    ragas_scores.append(row)

ragas_scores

C:\Users\rames\AppData\Local\Temp\ipykernel_18940\643046577.py:4: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import AnswerRelevancy, Faithfulness
C:\Users\rames\AppData\Local\Temp\ipykernel_18940\643046577.py:4: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import AnswerRelevancy, Faithfulness


[{'language': 'en',
  'query': 'How much overtime pay am I entitled to?',
  'answer': "To calculate your overtime pay, follow these steps:\n\n1. Determine if you are a non-workman (monthly basic salary ≤ $2,600) or workman (≤ $4,500). \n2. Calculate your hourly basic rate of pay:\n   - Monthly-rated employee: (12 × monthly basic rate) ÷ (52 × 44)\n   - Daily-rated employee: daily pay at the basic rate ÷ working hours per day\n   - Piece-rated employee: total weekly pay ÷ total weekly hours\n\n3. Multiply this hourly rate by 1.5 for overtime.\n\n4. If you worked more than half your normal daily working hours on a rest day, add your normal salary plus overtime (2 days' salary + overtime). Otherwise, it's either:\n   - 1 day's salary if employer requested\n   - Half day's salary if employee requested\n\nExample: Non-workman earning $2,600 monthly, works 2 overtime hours:\n\nHourly rate = ($2,600 × 12) ÷ (52 × 44) ≈ $8.15\nOvertime pay per hour = $8.15 × 1.5 = $12.24\nTotal overtime for 2 

## Step 6: Review results

Per-query faithfulness/answer-relevancy scores, full answer text on any
flagged row, then compare the averages against `06_eval.ipynb`'s qwen3:8b
numbers to see whether SEA-LION's generation is actually more accurate.

In [6]:
for row in ragas_scores:
    faithfulness = row["faithfulness"]
    relevancy = row["answer_relevancy"]
    flag = ""
    if faithfulness is None or faithfulness < 0.7:
        flag = "  <-- review"
    if relevancy is None or relevancy < 0.5:
        flag = "  <-- review"
    print(
        f"[{row['language']}] faithfulness={faithfulness}  answer_relevancy={relevancy}{flag}  "
        f"{row['query'][:50]}"
    )
    if flag:
        print(f"    answer: {row['answer']}")

valid_faithfulness = [r["faithfulness"] for r in ragas_scores if r["faithfulness"] is not None]
valid_relevancy = [r["answer_relevancy"] for r in ragas_scores if r["answer_relevancy"] is not None]

print(f"\nAvg faithfulness ({len(valid_faithfulness)}/{len(ragas_scores)} scored): "
      f"{sum(valid_faithfulness) / len(valid_faithfulness):.3f}" if valid_faithfulness else "\nNo faithfulness scores succeeded")
print(f"Avg answer_relevancy ({len(valid_relevancy)}/{len(ragas_scores)} scored): "
      f"{sum(valid_relevancy) / len(valid_relevancy):.3f}" if valid_relevancy else "No answer_relevancy scores succeeded")

[en] faithfulness=1.0  answer_relevancy=0.8770812008113374  How much overtime pay am I entitled to?
[en] faithfulness=0.7272727272727273  answer_relevancy=0.9130631596592256  When must my employer pay my salary?
[ms] faithfulness=1.0  answer_relevancy=0.8706802226895505  Bilakah majikan saya perlu bayar gaji saya?
[ta] faithfulness=1.0  answer_relevancy=0.7397435278716212  எனக்கு எவ்வளவு கூடுதல் நேர ஊதியம் கிடைக்கும்?
[vi] faithfulness=1.0  answer_relevancy=0.759774583562916  Chủ sử dụng lao động của tôi phải trả lương khi nà
[en] faithfulness=0.75  answer_relevancy=0.6998876542360936  Who pays repatriation costs when my Work Permit en
[en] faithfulness=0.8888888888888888  answer_relevancy=0.6411274629040523  How much medical insurance must my employer provid
[en] faithfulness=0.8333333333333334  answer_relevancy=0.9999999999999147  How can I contact MOM?

Avg faithfulness (8/8 scored): 0.900
Avg answer_relevancy (8/8 scored): 0.813
